# Smoke test: `opentelemetry_instrumentation_openai_v2`

Exercises the public API of the OpenTelemetry OpenAI instrumentation: the `OpenAIInstrumentor` class, its declared dependencies, and the instrument/uninstrument lifecycle that patches the `openai` client. No request is sent to OpenAI.

**Version note:** opentelemetry-instrumentation-openai-v2 2.4b0 imports `opentelemetry.util.genai.instruments`, which opentelemetry-util-genai renamed in 1.2b0. Until a newer openai-v2 release ships, install `"opentelemetry-util-genai<=1.1b0"` alongside it (all util-genai releases are pre-releases, so pip ignores a plain `<1.2` pin). The instrumentation also imports `httpx` directly, which recent `openai` releases no longer install, so add `httpx` as well.

In [ ]:
# Example 1: Verify the OpenAIInstrumentor class is exposed and instantiable.
try:
    from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
except ModuleNotFoundError as e:
    if e.name == 'opentelemetry.util.genai.instruments':
        raise ModuleNotFoundError(
            "opentelemetry-instrumentation-openai-v2 imports opentelemetry.util.genai.instruments, "
            "which opentelemetry-util-genai renamed in 1.2b0; install "
            "'opentelemetry-util-genai<=1.1b0' alongside it (see the note at the top of this notebook)."
        ) from e
    raise

instrumentor = OpenAIInstrumentor()
print(f"OpenAIInstrumentor instance: {instrumentor}")
print(f"Type: {type(instrumentor).__name__}")

# The instrumentor should have the standard BaseInstrumentor methods.
for method in ('instrument', 'uninstrument', 'instrumentation_dependencies'):
    assert hasattr(instrumentor, method), f"missing method: {method}"
    print(f"  has method: {method}")

assert isinstance(instrumentor, OpenAIInstrumentor)

In [ ]:
# Example 2: Check instrumentation_dependencies returns the expected openai pin spec.
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

instrumentor = OpenAIInstrumentor()
deps = instrumentor.instrumentation_dependencies()
print(f"instrumentation_dependencies() = {deps}")

# Should return a non-empty iterable of pip-style requirement strings referencing openai.
deps_list = list(deps)
assert len(deps_list) >= 1, "expected at least one dependency spec"
assert any('openai' in d.lower() for d in deps_list), f"expected openai in deps: {deps_list}"
print(f"\nFound openai dependency spec: {deps_list}")

In [ ]:
# Example 3: BaseInstrumentor inheritance chain check.
from opentelemetry.instrumentation.instrumentor import BaseInstrumentor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

assert issubclass(OpenAIInstrumentor, BaseInstrumentor)
print(f"OpenAIInstrumentor MRO:")
for cls in OpenAIInstrumentor.__mro__:
    print(f"  {cls.__module__}.{cls.__name__}")

# Singletonish: instantiate twice and confirm BaseInstrumentor's singleton pattern.
a = OpenAIInstrumentor()
b = OpenAIInstrumentor()
print(f"\na is b: {a is b} (BaseInstrumentor is a singleton)")
assert a is b

In [ ]:
# Example 4: instrument() + uninstrument() lifecycle without making a network call.
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

instrumentor = OpenAIInstrumentor()

# Start clean: if a previous cell instrumented, undo first.
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

assert not instrumentor.is_instrumented_by_opentelemetry
instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry
print("instrument() applied successfully")

instrumentor.uninstrument()
assert not instrumentor.is_instrumented_by_opentelemetry
print("uninstrument() reversed the patch cleanly")

In [ ]:
# Example 5: Patched openai.OpenAI.chat.completions.create after instrument().
import openai
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

instrumentor = OpenAIInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

original_create = openai.resources.chat.completions.Completions.create
instrumentor.instrument()
patched_create = openai.resources.chat.completions.Completions.create

print(f"original create: {original_create}")
print(f"patched create:  {patched_create}")
assert original_create is not patched_create, "instrument() should have replaced .create"

instrumentor.uninstrument()
restored_create = openai.resources.chat.completions.Completions.create
assert restored_create is original_create, "uninstrument() should restore original .create"
print("\nPatch + restore lifecycle verified")